In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Setup + data

In [2]:
!pip install -q peft
!pip uninstall -y torchao

import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

DATA    = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = list("ABCDE")

train = pd.read_csv(f"{DATA}/train.csv")
train["label"] = train["answer"].map({c: i for i, c in enumerate(OPTIONS)})
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Multiple-Choice Data Formatting**

*In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.*

**Q1. Label Encoding**

Convert the answer column in train.csv into numeric labels using the following mapping:

A = 0

B = 1

C = 2

D = 3

E = 4

What is the encoded numeric label for the row at index 150?

**Q2. Prompt-Option Formatting**

For row index 0, create the Option B input using exactly this format:

str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [3]:
Q1 = int(train.loc[150, "label"])

row0 = train.iloc[0]
formatted_B = str(row0["prompt"]) + " [SEP] " + str(row0["B"])
Q2 = len(formatted_B)
print("Q1:", Q1, "| Q2:", Q2)

Q1: 2 | Q2: 407


**Tokenization for Multiple-Choice Models**

*Multiple-choice models expect inputs in the shape:*

*batch_size x num_choices x sequence_length*

*Since each question has five options, every row becomes five tokenized sequences.*

**Q3.** **Single-Row MCQ Tokenization**

Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:

padding = "max_length"

truncation = True

max_length = 128

return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [4]:
inputs_row0 = [str(row0["prompt"]) + " [SEP] " + str(row0[o]) for o in OPTIONS]
enc0 = tokenizer(inputs_row0, padding="max_length", truncation=True,
                 max_length=128, return_tensors="pt")
input_ids_row0 = enc0["input_ids"].unsqueeze(0)
print("shape:", input_ids_row0.shape)
Q3 = input_ids_row0.shape[1]
print("Q3:", Q3)

shape: torch.Size([1, 5, 128])
Q3: 5


**Q4. Batch MCQ Tokenization**

Tokenize the first 16 rows of train.csv as multiple-choice examples.

Each row has 5 choices.

Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [5]:
batch_texts = []
for i in range(16):
    r = train.iloc[i]
    batch_texts += [str(r["prompt"]) + " [SEP] " + str(r[o]) for o in OPTIONS]

enc16 = tokenizer(batch_texts, padding="max_length", truncation=True,
                  max_length=128, return_tensors="pt")
ids16 = enc16["input_ids"].view(16, 5, 128)
print("shape:", ids16.shape)
Q4 = int(np.prod(ids16.shape))       
print("Q4:", Q4)

shape: torch.Size([16, 5, 128])
Q4: 10240


**Multiple-Choice Model Outputs**

*AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E*

**Q5. Multiple-Choice Logits**

Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question?

**Q6. Supervised Loss Tensor**

For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [6]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

with torch.no_grad():
    out = model(input_ids=input_ids_row0,
                attention_mask=enc0["attention_mask"].unsqueeze(0))
print("logits shape:", out.logits.shape)     
Q5 = out.logits.shape[1]

label0 = torch.tensor([int(row0["label"])])
out_l = model(input_ids=input_ids_row0,
              attention_mask=enc0["attention_mask"].unsqueeze(0),
              labels=label0)
Q6 = out_l.loss.dim()
print("Q5:", Q5, "| Q6:", Q6)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


logits shape: torch.Size([1, 5])
Q5: 5 | Q6: 0


**LoRA for Efficient Fine-Tuning**

*LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.*

**Q7. LoRA Trainable Parameters**

Apply LoRA to the bert-base-uncased multiple-choice model using:

r = 8

lora_alpha = 16

target_modules = ["query", "value"]

lora_dropout = 0.1

bias = "none"

task_type = TaskType.SEQ_CLS


Count trainable parameters using:

sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [7]:
lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1, bias="none",
    task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(AutoModelForMultipleChoice.from_pretrained("bert-base-uncased"),
                            lora_config)
Q7 = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Q7:", Q7)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q7: 295681


**Preparing Data for Hugging Face Trainer**

*Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.*

**Q8. Hugging Face Dataset Preparation**

Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:

input_ids with shape [5, 128]

attention_mask with shape [5, 128]

labels as the encoded answer label


For the first dataset item, input_ids has shape:
[5, 128]


How many tokenized choices are stored in input_ids?

In [8]:
def encode_row(r):
    texts = [str(r["prompt"]) + " [SEP] " + str(r[o]) for o in OPTIONS]
    enc = tokenizer(texts, padding="max_length", truncation=True, max_length=128)
    return {"input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": int(r["label"])}

records = [encode_row(train.iloc[i]) for i in range(100)]
ds100 = Dataset.from_list(records)
first = ds100[0]
print("input_ids shape:", np.array(first["input_ids"]).shape)   
Q8 = np.array(first["input_ids"]).shape[0]
print("Q8:", Q8)

input_ids shape: (5, 128)
Q8: 5


**Tiny Fine-Tuning and Inference**

*In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.*

**Q9. Tiny LoRA Fine-Tuning**

Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:

max_length = 64

per_device_train_batch_size = 4

gradient_accumulation_steps = 1

max_steps = 4

What is the final global_step reported by the Trainer?

In [9]:
def encode_row64(r):
    texts = [str(r["prompt"]) + " [SEP] " + str(r[o]) for o in OPTIONS]
    enc = tokenizer(texts, padding="max_length", truncation=True, max_length=64)
    return {"input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": int(r["label"])}

ds32 = Dataset.from_list([encode_row64(train.iloc[i]) for i in range(32)])
ds32.set_format("torch")

args = TrainingArguments(
    output_dir="m4_out",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    report_to="none",
    logging_steps=1,
)
trainer = Trainer(model=lora_model, args=args, train_dataset=ds32)
result = trainer.train()
Q9 = trainer.state.global_step
print("Q9:", Q9)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,3.172438
2,3.225303
3,3.266438
4,3.239956


Q9: 4


**Q10. Probability Assigned to Option E After Fine-Tuning**

Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [10]:
lora_model.eval()
device = next(lora_model.parameters()).device
with torch.no_grad():
    out = lora_model(input_ids=input_ids_row0.to(device),
                     attention_mask=enc0["attention_mask"].unsqueeze(0).to(device))
probs = torch.softmax(out.logits, dim=-1)[0]
Q10 = round(float(probs[4]), 4)
print("probs:", [round(float(p), 4) for p in probs])
print("Q10:", Q10)

probs: [0.199, 0.1978, 0.2029, 0.1977, 0.2026]
Q10: 0.2026


In [11]:
print("MILESTONE 4 - ANSWERS")
for k, v in dict(Q1=Q1, Q2=Q2, Q3=Q3, Q4=Q4, Q5=Q5,
                 Q6=Q6, Q7=Q7, Q8=Q8, Q9=Q9, Q10=Q10).items():
    print(f"{k}: {v}")

MILESTONE 4 - ANSWERS
Q1: 2
Q2: 407
Q3: 5
Q4: 10240
Q5: 5
Q6: 0
Q7: 295681
Q8: 5
Q9: 4
Q10: 0.2026
